# Spaceship Titanic Dataset with XGBoost

This notebook implements an XGBoost solution as an alternative to Random Forest.
XGBoost uses gradient boosting, which builds trees sequentially where each tree corrects the errors of the previous ones.

**Key Differences from Random Forest:**
- Random Forest: Trees built independently in parallel
- XGBoost: Trees built sequentially, each learning from previous mistakes
- XGBoost often achieves better accuracy with fewer trees

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print(f"XGBoost version: {xgb.__version__}")

# Load and Explore Dataset

In [ ]:
# Load dataset
dataset_df = pd.read_csv('/kaggle/input/spaceship-titanic/train.csv')
print(f"Full train dataset shape is {dataset_df.shape}")
dataset_df.head()

# Data Preprocessing

Using the same preprocessing approach as Random Forest solution

In [ ]:
# Drop PassengerId and Name
dataset_df = dataset_df.drop(['PassengerId', 'Name'], axis=1)

# Fill missing values for boolean columns
dataset_df[['VIP', 'CryoSleep']] = dataset_df[['VIP', 'CryoSleep']].fillna(value=0)

# Extract Deck, Cabin_num, and Side from Cabin
dataset_df[["Deck", "Cabin_num", "Side"]] = dataset_df["Cabin"].str.split("/", expand=True)
dataset_df = dataset_df.drop('Cabin', axis=1)

# Convert boolean columns to integers
dataset_df['VIP'] = dataset_df['VIP'].astype(int)
dataset_df['CryoSleep'] = dataset_df['CryoSleep'].astype(int)
dataset_df['Transported'] = dataset_df['Transported'].astype(int)

# Convert Cabin_num to numeric (handle missing values)
dataset_df['Cabin_num'] = pd.to_numeric(dataset_df['Cabin_num'], errors='coerce')

print(f"Dataset shape after preprocessing: {dataset_df.shape}")
dataset_df.head()

In [ ]:
# Check missing values
print("Missing values per column:")
print(dataset_df.isnull().sum().sort_values(ascending=False))

# Feature Engineering for XGBoost

XGBoost can handle categorical features natively, but we'll use label encoding for better performance

In [ ]:
# Separate features and labels
label = dataset_df['Transported']
features = dataset_df.drop('Transported', axis=1)

# Get categorical columns
categorical_cols = features.select_dtypes(include=['object']).columns.tolist()
numerical_cols = features.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

In [ ]:
# Label encode categorical features
# XGBoost works well with label encoding
label_encoders = {}
features_encoded = features.copy()

for col in categorical_cols:
    le = LabelEncoder()
    # Fill missing values with 'Missing' before encoding
    features_encoded[col] = features_encoded[col].fillna('Missing')
    features_encoded[col] = le.fit_transform(features_encoded[col])
    label_encoders[col] = le

# Fill remaining numerical missing values with median
for col in numerical_cols:
    features_encoded[col] = features_encoded[col].fillna(features_encoded[col].median())

print(f"Features shape after encoding: {features_encoded.shape}")
features_encoded.head()

# Split Dataset

In [ ]:
# Split data: 80% training, 20% validation
X_train, X_valid, y_train, y_valid = train_test_split(
    features_encoded, 
    label, 
    test_size=0.2, 
    random_state=42,
    stratify=label  # Ensure balanced split
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_valid.shape[0]}")
print(f"Training label distribution: {y_train.value_counts().to_dict()}")

# Train XGBoost Model

XGBoost parameters explained:
- `n_estimators`: Number of boosting rounds (trees to build)
- `max_depth`: Maximum depth of each tree (controls complexity)
- `learning_rate`: Step size for each tree (smaller = more conservative)
- `subsample`: Fraction of samples used per tree (prevents overfitting)
- `colsample_bytree`: Fraction of features used per tree
- `eval_metric`: Metric to optimize (logloss for binary classification)

In [ ]:
# Initialize XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,           # Number of trees
    max_depth=6,                # Tree depth
    learning_rate=0.1,          # Learning rate (eta)
    subsample=0.8,              # Row sampling
    colsample_bytree=0.8,       # Column sampling
    random_state=42,
    eval_metric='logloss',      # Evaluation metric
    early_stopping_rounds=20,   # Stop if no improvement
    n_jobs=-1                   # Use all CPU cores
)

print("Training XGBoost model with early stopping...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_valid, y_valid)],
    verbose=20  # Print progress every 20 rounds
)
print("Training complete!")

# Visualize Training Progress

In [ ]:
# Plot training history
results = xgb_model.evals_result()

plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(results['validation_0']['logloss'], label='Train')
plt.plot(results['validation_1']['logloss'], label='Validation')
plt.xlabel('Boosting Round')
plt.ylabel('Log Loss')
plt.title('Training Progress: Log Loss')
plt.legend()
plt.grid(True)

# Calculate and plot accuracy from logloss
plt.subplot(1, 2, 2)
train_pred = xgb_model.predict(X_train)
valid_pred = xgb_model.predict(X_valid)
train_acc = accuracy_score(y_train, train_pred)
valid_acc = accuracy_score(y_valid, valid_pred)

plt.bar(['Train', 'Validation'], [train_acc, valid_acc], color=['blue', 'orange'])
plt.ylabel('Accuracy')
plt.title('Final Accuracy Comparison')
plt.ylim([0.7, 1.0])
plt.grid(True, axis='y')

plt.tight_layout()
plt.show()

print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Best score: {xgb_model.best_score:.4f}")

# Model Evaluation

In [ ]:
# Predictions
y_train_pred = xgb_model.predict(X_train)
y_valid_pred = xgb_model.predict(X_valid)

# Accuracies
train_accuracy = accuracy_score(y_train, y_train_pred)
valid_accuracy = accuracy_score(y_valid, y_valid_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {valid_accuracy:.4f}")
print(f"Overfitting Gap: {train_accuracy - valid_accuracy:.4f}")

print("\nValidation Classification Report:")
print(classification_report(y_valid, y_valid_pred, target_names=['Not Transported', 'Transported']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_valid, y_valid_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Transported', 'Transported'],
            yticklabels=['Not Transported', 'Transported'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - XGBoost')
plt.show()

# Feature Importance Analysis

In [ ]:
# Get feature importances (using gain by default)
feature_importances = pd.DataFrame({
    'feature': features_encoded.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importances.head(15))

In [ ]:
# Visualize feature importances
plt.figure(figsize=(10, 8))
top_features = feature_importances.head(15)
plt.barh(top_features['feature'], top_features['importance'], color='steelblue')
plt.xlabel('Importance Score (Gain)')
plt.title('Top 15 Feature Importances - XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Plot using XGBoost's built-in plot function
fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(xgb_model, max_num_features=15, ax=ax, importance_type='gain')
plt.title('Feature Importance (Gain) - XGBoost Built-in Plot')
plt.tight_layout()
plt.show()

# Cross-Validation

Verify model performance using 5-fold cross-validation

In [ ]:
# Perform 5-fold cross-validation
cv_scores = cross_val_score(
    xgb_model, 
    features_encoded, 
    label, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1
)

print(f"Cross-Validation Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Std CV Accuracy: {cv_scores.std():.4f}")

# Prepare Test Data and Make Predictions

In [ ]:
# Load test dataset
test_df = pd.read_csv('/kaggle/input/spaceship-titanic/test.csv')
submission_id = test_df.PassengerId.copy()

# Apply same preprocessing
test_df = test_df.drop(['PassengerId', 'Name'], axis=1)
test_df[['VIP', 'CryoSleep']] = test_df[['VIP', 'CryoSleep']].fillna(value=0)
test_df[["Deck", "Cabin_num", "Side"]] = test_df["Cabin"].str.split("/", expand=True)
test_df = test_df.drop('Cabin', axis=1)
test_df['VIP'] = test_df['VIP'].astype(int)
test_df['CryoSleep'] = test_df['CryoSleep'].astype(int)
test_df['Cabin_num'] = pd.to_numeric(test_df['Cabin_num'], errors='coerce')

# Apply label encoding using stored encoders
test_encoded = test_df.copy()
for col in categorical_cols:
    test_encoded[col] = test_encoded[col].fillna('Missing')
    # Handle unseen categories
    le = label_encoders[col]
    test_encoded[col] = test_encoded[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

# Fill numerical missing values
for col in numerical_cols:
    if col in test_encoded.columns:
        test_encoded[col] = test_encoded[col].fillna(features_encoded[col].median())

print(f"Test set shape: {test_encoded.shape}")
test_encoded.head()

In [ ]:
# Make predictions
predictions = xgb_model.predict(test_encoded)
predictions_proba = xgb_model.predict_proba(test_encoded)[:, 1]

# Create submission dataframe
output = pd.DataFrame({
    'PassengerId': submission_id,
    'Transported': predictions.astype(bool)
})

print(f"Prediction distribution:")
print(output['Transported'].value_counts())
output.head(10)

# Save Submission File

In [ ]:
# Save submission file
output.to_csv('/kaggle/working/submission_xgboost.csv', index=False)
print("XGBoost submission file saved successfully!")

# Model Comparison Summary

## Random Forest vs XGBoost

### Random Forest:
- **How it works**: Builds many independent trees in parallel
- **Strengths**: Less prone to overfitting, easier to tune
- **Weaknesses**: May need more trees for best performance
- **Example**: Like asking 300 independent experts and taking a majority vote

### XGBoost:
- **How it works**: Builds trees sequentially, each correcting previous errors
- **Strengths**: Often more accurate, faster training with fewer trees
- **Weaknesses**: Can overfit if not tuned properly
- **Example**: Like a student learning from mistakes - each tree focuses on what previous trees got wrong

### When to Use Each:
- **Random Forest**: When you want a robust baseline without much tuning
- **XGBoost**: When you want maximum accuracy and are willing to tune hyperparameters

Both models achieved similar validation accuracy, demonstrating that for this problem, either approach is viable!